In [ ]:
import torch

ModuleNotFoundError: No module named 'torch'

In [1]:
pip install torch pandas scikit-learn sentence-transformers tqdm

  Using cached torch-2.8.0-cp313-none-macosx_11_0_arm64.whl.metadata (30 kB)
  Using cached pandas-2.3.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached scikit_learn-1.7.1-cp313-cp313-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached sentence_transformers-5.1.0-py3-none-any.whl.metadata (16 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.7.0-py3-none-any.whl.metadata (12 kB)
  Using cached numpy-2.3.2-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.16.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (61 kB)
  Usi

In [2]:
pip install --upgrade pip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.7 kB/s eta 0:00:0000:0100:02m
  Attempting uninstall: pip
    Found existing installation: pip 23.0.1
    Uninstalling pip-23.0.1:
      Successfully uninstalled pip-23.0.1
Note: you may need to restart the kernel to use updated packages.


In [2]:
import math, gc, torch, torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm


class Args: pass
args = Args()
args.data   = "merged_with_embeddings.parquet"
args.past   = 20       # past days
args.future = 5        # future days
args.batch  = 256      # GPU adjustion
args.epochs = 20
args.lr     = 1e-4
args.d_txt  = 768      # embedding dimension (MiniLM=384, GTE/e5=768, Qwen=1024)


d_txt = len(df_raw.loc[0, "passage_emb"])   # 自动探测
print("Detected embedding dim:", d_txt)


device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| d_txt:", args.d_txt)

# Dataset
class ECPriceDataset(Dataset):
    def __init__(self, df, past_days=20, future_days=5):
        self.df = df.reset_index(drop=True)
        self.past = past_days
        self.future = future_days

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x_txt   = torch.tensor(row.passage_emb, dtype=torch.float32)
        x_price = torch.tensor(row.X_price,    dtype=torch.float32)  # (past,1)
        y       = torch.tensor(row.y,          dtype=torch.float32)  # (future,)
        return x_price, x_txt, y


class DualModalCrossAttn(nn.Module):
    def __init__(self, d_price_in=1, d_txt=384, d_model=128,
                 n_heads=4, seq_len=20, out_days=5):
        super().__init__()
        # 价格分支
        self.pos   = nn.Embedding(seq_len, d_model)
        self.p_proj= nn.Linear(d_price_in, d_model)
        self.p_attn= nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        # 文本分支
        self.t_proj= nn.Sequential(
            nn.Linear(d_txt, d_model), nn.ReLU(),
            nn.Linear(d_model, d_model))
        # Cross-Attention
        self.cross = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        # 额外 Transformer
        enc_layer = nn.TransformerEncoderLayer(d_model, n_heads,
                                               4*d_model, batch_first=True)
        self.fuser = nn.TransformerEncoder(enc_layer, num_layers=2)
        # 预测头
        self.head  = nn.Linear(d_model, out_days)

    def forward(self, price_seq, txt_emb):
        B, T, _ = price_seq.shape
        pos = self.pos(torch.arange(T, device=price_seq.device))[None]
        h_p = self.p_proj(price_seq) + pos                 # (B,T,d)
        h_p, _ = self.p_attn(h_p, h_p, h_p)                # self-attn
        z_p = h_p.mean(dim=1, keepdim=True)                # (B,1,d)

        z_t = self.t_proj(txt_emb).unsqueeze(1)            # (B,1,d)
        z, _ = self.cross(z_p, z_t, z_t)                   # (B,1,d)
        z = self.fuser(z)                                  # (B,1,d)
        return self.head(z.squeeze(1))                     # (B,future)

# ============================================================
# 4. 构造样本
# ============================================================
def build_samples(df_raw, past=20, future=5):
    df = df_raw.copy()
    df["Date"] = pd.to_datetime(df["Date"])

    samples = []
    for call_id, g in df.groupby("id"):         
        g = g.sort_values("Date").reset_index(drop=True)

        prices = g["Price"].values
        emb    = g["passage_emb"].values[0]

        for i in range(past, len(g) - future):
            past_slice   = prices[i-past:i]
            future_slice = prices[i+1:i+1+future]
            if len(future_slice) < future:
                continue
            samples.append({
                "passage_emb": emb,
                "X_price": past_slice.reshape(-1, 1),
                "y": future_slice
            })

    return pd.DataFrame(samples)


# ============================================================
# 5. 训练/验证步骤
# ============================================================
def step(loader, model, loss_fn, optimizer=None, scaler=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total, total_loss = 0, 0.0
    with torch.set_grad_enabled(train):
        for x_p, x_t, y in loader:
            x_p, x_t, y = x_p.to(device), x_t.to(device), y.to(device)
            if train and scaler:
                with torch.autocast("cuda"):
                    pred = model(x_p, x_t)
                    loss = loss_fn(pred, y)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            elif train:
                pred = model(x_p, x_t); loss = loss_fn(pred, y)
                loss.backward(); optimizer.step(); optimizer.zero_grad()
            else:
                pred = model(x_p, x_t); loss = loss_fn(pred, y)
            total += y.size(0); total_loss += loss.item()*y.size(0)
    return total_loss/total

# ============================================================
# 6. 主流程
# ============================================================
# 读取数据
df_raw = pd.read_parquet(args.data)
df_samples = build_samples(df_raw, args.past, args.future)
print("样本量:", len(df_samples))

# train / val 划分
tr_df, val_df = train_test_split(df_samples, test_size=0.2, random_state=42)
train_dl = DataLoader(ECPriceDataset(tr_df, args.past, args.future),
                      batch_size=args.batch, shuffle=True, drop_last=True)
val_dl   = DataLoader(ECPriceDataset(val_df, args.past, args.future),
                      batch_size=args.batch)

# 初始化模型
d_txt = len(df_raw.loc[0, "passage_emb"])
model = DualModalCrossAttn(
    d_price_in = 1,
    d_txt      = d_txt,     # 使用自动探测的维度
    d_model    = 128,
    n_heads    = 4,
    seq_len    = args.past,
    out_days   = args.future
).to(device)

opt    = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
loss_f = nn.MSELoss()
scaler = torch.cuda.amp.GradScaler() if device=="cuda" else None

best = math.inf
for epoch in range(1, args.epochs+1):
    tr_loss = step(train_dl, model, loss_f, opt, scaler)
    val_loss = step(val_dl, model, loss_f)
    if val_loss < best:
        best = val_loss; torch.save(model.state_dict(), "best_cross_attn.pt")
    print(f"[{epoch:02d}] train {tr_loss:.4f} | val {val_loss:.4f} | best {best:.4f}")

print("Best Validation MSE:", best)

ModuleNotFoundError: No module named 'torch'